In [3]:
%pylab inline 

import torch 
import torch.nn as nn
import torch.nn.functional as F 
import torchvision
from torchvision import transforms 
from tqdm import trange

Populating the interactive namespace from numpy and matplotlib


In [4]:
# Device Options 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Data
norm_values = (0.5, 0.5, 0.5) 
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(norm_values,norm_values)
    ])
trainset = torchvision.datasets.CIFAR10(root="datasets/",download=False, train=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root="datasets/", download=False, train=False, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, shuffle=True, batch_size=16)
testloader = torch.utils.data.DataLoader(testset, shuffle=False, batch_size=16)



In [11]:
# Write Training function to test on different models 
def train(model, epochs=5):
    model.train()
    # Loss And Optimization 
    loss_function = nn.CrossEntropyLoss()
    optim = torch.optim.Adam(params=model.parameters())

    # Training Loop 
    for _ in (t := trange(epochs)):
        for x,y in trainloader:
            x,y = x.to(device), y.to(device)

            # Forward Pass 
            outputs = model(x)
            loss = loss_function(outputs, y)

            # Update Weights
            optim.zero_grad()
            loss.backward()
            optim.step()

            # Training Accuracy 
            n_correct = torch.argmax(outputs, dim=1)
            acc = (n_correct == y).float().mean() * 100
        
        t.set_description(f"acc = {acc.item():.2f}%, loss = {loss.item():.2f}")
    return model
        


      


In [6]:
# Test Model Accuracy on test set
def show_model_accuracy(model):
    model.eval()
    with torch.no_grad():
        for x,y in testloader:
            x,y = x.to(device), y.to(device)
            outputs = model(x)
            n_correct = torch.argmax(outputs, dim=1)
            acc = (n_correct == y).float().mean()*100
        print(f"Val Accuracy = {acc.item():.2f}%")
     





In [7]:
# Model Evaluation 
def eval_model(model):
    trained_model = train(model)
    acc = show_model_accuracy(trained_model)
    return acc 


In [8]:
# Model with basic convnet architecture  
class BasicNet(nn.Module):
    def __init__(self):
        super(BasicNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 12, 5) # 16, 3, 28, 28 
        self.pool = nn.MaxPool2d(2,2) # 16, 12, 14, 14 
        self.conv2 = nn.Conv2d(12, 24, 5) # 16, 24, 10, 10 
        self.fc1 = nn.Linear(24*10*10, 128) 
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = x.view(-1, 24*10*10)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x 




In [12]:
# How good is our basic convnet?
bnet = BasicNet().to(device)
eval_model(bnet)

acc = 75.00%, loss = 0.68: 100%|██████████| 5/5 [01:02<00:00, 12.53s/it]
Val Accuracy = 56.25%


In [14]:
# AlexNet Model
# Paper: https://proceedings.neurips.cc/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf
# Code:https://github.com/icpm/pytorch-cifar10/blob/master/models/AlexNet.py

class AlexNet(nn.Module):
    def __init__(self):
        super(AlexNet, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)
        )
        self.lin_layers = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 2 * 2, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, 10)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), 256*2*2)
        x = self.lin_layers(x)
        return x 

    

    

In [15]:
# How Good is AlexNet?
alexnet = AlexNet().to(device)
eval_model(alexnet)

acc = 50.00%, loss = 1.40: 100%|██████████| 5/5 [01:56<00:00, 23.35s/it]
Val Accuracy = 62.50%
